In [41]:
import pandas as pd
import numpy as np
import sklearn
print("All libraries loaded successfully!")

All libraries loaded successfully!


In [42]:
import os
import urllib.request

# Define the URL for the GSE16879 Series Matrix file from NCBI GEO
data_url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE16nnn/GSE16879/matrix/GSE16879_series_matrix.txt.gz"

# Define the local path where we want to save the file
dest_path = os.path.join("data", "GSE16879_series_matrix.txt.gz")

# Check if the file already exists locally so we don't re-download it every time
if not os.path.exists(dest_path):
    print("Downloading dataset from NCBI GEO... This might take a minute.")
    urllib.request.urlretrieve(data_url, dest_path)
    print(f"Download complete! File saved to: {dest_path}")
else:
    print("Dataset already exists locally in the 'data' folder.")

Dataset already exists locally in the 'data' folder.


In [ ]:
#caused crash, not using this
#import gzip
#import os

#file_path = os.path.join("data", "GSE16879_series_matrix.txt.gz")

# 'rt' means Read Text. gzip handles unzipping the file into RAM without creating permanent unzipped file in storage
#with gzip.open(file_path, 'rt') as f:
#    for i in range(30):
#        line = f.readline()
#        # Print the line number (1-30 rather than 0-29) and the raw text inside that line
#        print(f"Line {i+1}: {line.strip()}")

In [43]:
import gzip
import os

file_path = os.path.join("data", "GSE16879_series_matrix.txt.gz")

# 'rt' means read text. We add explicit encoding and tell it to ignore decoding errors
with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
    for i in range(40):
        line = f.readline().strip()
        # Show the first 120 characters of each line so it doesn't wrap messily
        print(f"Line {i+1}: {line[:120]}")

Line 1: !Series_title	"Mucosal expression profiling in patients with inflammatory bowel disease before and after first inflixima
Line 2: !Series_geo_accession	"GSE16879"
Line 3: !Series_status	"Public on Nov 03 2009"
Line 4: !Series_submission_date	"Jun 29 2009"
Line 5: !Series_last_update_date	"Mar 25 2019"
Line 6: !Series_pubmed_id	"19956723"
Line 7: !Series_summary	"We used microarrays to identify mucosal gene signatures predictive of response to infliximab (IFX) in p
Line 8: !Series_summary	""
Line 9: !Series_summary	"Keywords: drug response and treatment effect"
Line 10: !Series_overall_design	"Mucosal biopsies were obtained at endoscopy in actively inflamed mucosa from 61 IBD patients (24
Line 11: !Series_type	"Expression profiling by array"
Line 12: !Series_contributor	"Ingrid,,Arijs"
Line 13: !Series_contributor	"Leentje,,Van Lommel"
Line 14: !Series_contributor	"Kristel,,Van Steen"
Line 15: !Series_contributor	"Gert,,De Hertogh"
Line 16: !Series_contributor	"Karel,,Geboes"
Lin

In [ ]:
import gzip
import os
import pandas as pd

file_path = os.path.join("data", "GSE16879_series_matrix.txt.gz")

metadata_dict = {}

# 1. Parse the file line-by-line to collect sample metadata
with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
    for line in f:
        # If we hit the expression matrix, stop reading metadata
        if not line.startswith("!"):
            if line.strip() and not line.startswith('"ID_REF"'):
                break # We reached the numeric data zone
                
        # Capture only the sample-specific metadata rows
        if line.startswith("!Sample_"):
            parts = line.strip().split("\t")
            key = parts[0]       # e.g., "!Sample_title"
            values = parts[1:]   # All the patient values for that row
            metadata_dict[key] = values

# 2. Convert our dictionary into a Pandas DataFrame
df_meta = pd.DataFrame.from_dict(metadata_dict)

# 3. Transpose the dataframe so patients are rows, and traits are columns
df_meta = df_meta.T

# 4. Clean up the index names for readability
df_meta.columns = [c.replace('"', '').strip() for c in df_meta.iloc[0]] # Clean column strings
# Let's peek at the first 5 rows and first 4 columns of our new table
df_meta.iloc[:5, :4]

In [ ]:
#transpose to make patients the rows, and the traits the collumns
df_clinical = df_meta.T 

#remove the exclamation mark and 'Sample_' prefix
df_clinical.columns = [col.replace('!Sample_', ' ').strip() for col in df_clinical.columns]

#read first 5 rows and 5 columns
df_clinical.iloc[:5, :5]

In [ ]:
#print a list of the columns
list(df_clinical.columns)

In [ ]:
list(df_clinical.index)

In [ ]:
#display every column header in dataframe
for index, col_name in enumerate (df_clinical.columns):
    print(f"Column {index}: {col_name}")

In [ ]:
df_clinical['characteristics_ch1'].head(10)

In [44]:
import gzip
import os
import pandas as pd

file_path = os.path.join("data", "GSE16879_series_matrix.txt.gz")
metadata_dict = {}

with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
    for line in f:
        # Stop reading when we hit the numeric data
        if not line.startswith("!"):
            if line.strip() and not line.startswith('"ID_REF"'):
                break 
                
        if line.startswith("!Sample_"):
            parts = line.strip().split("\t")
            
            # 1. Clean the key (e.g., "title")
            base_key = parts[0].replace('!Sample_', '').strip()
            
            # 2. Clean the values (the patient data list)
            values = [v.replace('"', '').strip() for v in parts[1:]]
            final_key = base_key # Start with the original name (e.g., "characteristics_ch1")
            
            counter = 1          # Start our number tag at 1

            # WHILE the name is already taken in the dictionary...
            while final_key in metadata_dict:
                # ...create a new name by gluing the counter to the original name
                final_key = f"{base_key}_{counter}"
                # ...and increase the counter by 1 for the next loop (if needed)
                counter +=1
            # ==========================================
            
            # 3. Save the data safely into the dictionary
            metadata_dict[final_key] = values

# Convert to DataFrame and Transpose
df_clinical = pd.DataFrame.from_dict(metadata_dict).T
list(df_clinical.columns)

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132]

In [45]:
list(df_clinical.index)

['title',
 'geo_accession',
 'status',
 'submission_date',
 'last_update_date',
 'type',
 'channel_count',
 'source_name_ch1',
 'organism_ch1',
 'characteristics_ch1',
 'characteristics_ch1_1',
 'characteristics_ch1_2',
 'characteristics_ch1_3',
 'molecule_ch1',
 'extract_protocol_ch1',
 'label_ch1',
 'label_protocol_ch1',
 'taxid_ch1',
 'hyb_protocol',
 'scan_protocol',
 'description',
 'data_processing',
 'platform_id',
 'contact_name',
 'contact_email',
 'contact_laboratory',
 'contact_department',
 'contact_institute',
 'contact_address',
 'contact_city',
 'contact_zip/postal_code',
 'contact_country',
 'supplementary_file',
 'data_row_count',
 'relation']

In [46]:
# 1. Flip the table and overwrite the old one so it's permanent
df_clinical = df_clinical.T

# 2. Slice out just our four characteristics columns, and peek at the top 5 rows
df_clinical[['characteristics_ch1', 'characteristics_ch1_1', 'characteristics_ch1_2', 'characteristics_ch1_3']].head(5)

,characteristics_ch1,characteristics_ch1_1,characteristics_ch1_2,characteristics_ch1_3
0,tissue: Colon,disease: Control,response to infliximab: Not applicable,before or after first infliximab treatment: No...
1,tissue: Colon,disease: Control,response to infliximab: Not applicable,before or after first infliximab treatment: No...
2,tissue: Colon,disease: Control,response to infliximab: Not applicable,before or after first infliximab treatment: No...
3,tissue: Colon,disease: Control,response to infliximab: Not applicable,before or after first infliximab treatment: No...
4,tissue: Colon,disease: Control,response to infliximab: Not applicable,before or after first infliximab treatment: No...


In [48]:
#how many of each class are in a column
df_clinical['characteristics_ch1_2'].value_counts()

characteristics_ch1_2
response to infliximab: No                66
response to infliximab: Yes               55
response to infliximab: Not applicable    12
Name: count, dtype: int64

In [49]:
#removing the controls from the IBD dataframe
df_ibd = df_clinical[ df_clinical['characteristics_ch1_2'] != "response to infliximab: Not applicable"]

#clarifying that controls were removed
df_ibd['characteristics_ch1_2'].head(5)

6     response to infliximab: Yes
7     response to infliximab: Yes
8     response to infliximab: Yes
9     response to infliximab: Yes
10    response to infliximab: Yes
Name: characteristics_ch1_2, dtype: str

In [50]:
df_ibd['characteristics_ch1_2'] = df_ibd['characteristics_ch1_2'].str.replace("response to infliximab: ", "" )
df_ibd['characteristics_ch1_2'].head(5)


6     Yes
7     Yes
8     Yes
9     Yes
10    Yes
Name: characteristics_ch1_2, dtype: str

In [51]:
df_ibd['characteristics_ch1_2'].value_counts()

characteristics_ch1_2
No     66
Yes    55
Name: count, dtype: int64

In [ ]:
df_ibd[['characteristics_ch1', 'characteristics_ch1_1', 'characteristics_ch1_2', 'characteristics_ch1_3']].head(5)

In [ ]:
#remove unnecessary text
df_ibd['characteristics_ch1'] = df_ibd['characteristics_ch1'].str.replace("tissue: ", "")
df_ibd['characteristics_ch1_1'] = df_ibd['characteristics_ch1_1'].str.replace("disease: ", "")
df_ibd['characteristics_ch1_3'] = df_ibd['characteristics_ch1_1'].str.replace("before or after first infliximab treatment: ", "")


In [ ]:

df_ibd = df_ibd.rename(columns={'characteristics_ch1_1' : 'disease'})

In [ ]:
list(df_ibd.columns)

In [ ]:
df_ibd[['tissue', 'disease', 'response to infliximab', 'before or after first infliximab treatment']].head(5)

In [ ]:
import gzip
import os
import pandas as pd

# 1. PARSE THE RAW FILE
file_path = os.path.join("data", "GSE16879_series_matrix.txt.gz")
metadata_dict = {}

with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
    for line in f:
        if not line.startswith("!"):
            if line.strip() and not line.startswith('"ID_REF"'):
                break 
                
        if line.startswith("!Sample_"):
            parts = line.strip().split("\t")
            base_key = parts[0].replace('!Sample_', '').strip()
            values = [v.replace('"', '').strip() for v in parts[1:]]
            
            key = base_key
            counter = 1
            while key in metadata_dict:
                key = f"{base_key}_{counter}"
                counter += 1
                
            metadata_dict[key] = values

# 2. BUILD THE DATAFRAME (The rogue .T is gone!)
df_clinical = pd.DataFrame.from_dict(metadata_dict)

# 3. RENAME THE COLUMNS
df_clinical = df_clinical.rename(columns={
    'characteristics_ch1': 'tissue',
    'characteristics_ch1_2': 'Response to infliximab',
    'characteristics_ch1_3': 'Timing'
})

# 4. FILTER OUT CONTROLS
df_ibd = df_clinical[df_clinical['Response to infliximab'] != "response to infliximab: Not applicable"].copy()

# 5. CLEAN THE RESPONSE TEXT
df_ibd['Response to infliximab'] = df_ibd['Response to infliximab'].str.replace("response to infliximab:", "")

print("Reset Complete! Current columns are:")
print(list(df_ibd.columns))

In [ ]:
df_ibd['characteristics_ch1_1'].value_counts()

In [ ]:
df_ibd = df_ibd.rename(columns={'characteristics_ch1_1' : 'disease'})
df_ibd['Response to infliximab'] = df_ibd['Response to infliximab'].str.strip() # removes any spaces from that column

In [ ]:
df_ibd[['tissue', 'disease', 'Response to infliximab', 'Timing']].head(15)

In [ ]:
#placeholder
#df_ibd = df_ibd['tissue'].str.replace("tissue: ", "") already done
#df_ibd = df_ibd['disease'].str.replace("disease: ", "")
#df_ibd = df_ibd['Timing'].str.replace("before or after first infliximab treatment: ", "")


In [ ]:
#starting fresh
import gzip
import os
import pandas as pd

# 1. PARSE THE RAW FILE
file_path = os.path.join("data", "GSE16879_series_matrix.txt.gz")
metadata_dict = {}

with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
    for line in f:
        if not line.startswith("!"):
            if line.strip() and not line.startswith('"ID_REF"'):
                break 
                
        if line.startswith("!Sample_"):
            parts = line.strip().split("\t")
            base_key = parts[0].replace('!Sample_', '').strip()
            values = [v.replace('"', '').strip() for v in parts[1:]]
            
            key = base_key
            counter = 1
            while key in metadata_dict:
                key = f"{base_key}_{counter}"
                counter += 1
                
            metadata_dict[key] = values

# 2. BUILD THE DATAFRAME (The rogue .T is gone!)
df_clinical = pd.DataFrame.from_dict(metadata_dict)

# 3. RENAME THE COLUMNS
df_clinical = df_clinical.rename(columns={
    'characteristics_ch1': 'tissue',
    'characteristics_ch1_2': 'Response to infliximab',
    'characteristics_ch1_3': 'Timing'
})

# 4. FILTER OUT CONTROLS
df_ibd = df_clinical[df_clinical['Response to infliximab'] != "response to infliximab: Not applicable"].copy()

# 5. CLEAN THE RESPONSE TEXT
df_ibd['Response to infliximab'] = df_ibd['Response to infliximab'].str.replace("response to infliximab:", "")

print("Reset Complete! Current columns are:")
print(list(df_ibd.columns))

In [ ]:
df_ibd = df_ibd.rename(columns={'characteristics_ch1_1' : 'disease'})
df_ibd['Response to infliximab'] = df_ibd['Response to infliximab'].str.strip() # removes any spaces from that column

In [ ]:
#removing redundant strings
df_ibd['tissue'] = df_ibd['tissue'].str.replace("tissue: ", "").str.strip()
df_ibd['disease'] = df_ibd['disease'].str.replace("disease: ", "").str.strip()
df_ibd['Timing'] = df_ibd['Timing'].str.replace("before or after first infliximab treatment: ", "").str.strip()

#checking
df_ibd[['tissue', 'disease', 'Timing', 'Response to infliximab']].head()

In [ ]:
df_genes = pd.read_csv(
    file_path , #file defined earler
    sep='\t',   # columns are separated by tabs
    compression=  'gzip', #unzips file
    comment = '!', #ignores clinical header
    index_col = 0 #Gene IDs become row names
)

In [ ]:
df_genes.shape #rows = genes, columns = patiens

In [ ]:
df_genes = df_genes.T # transpose dataframe
df_genes = df_genes.loc[df_ibd.index]


In [ ]:
# 1. Fix the clinical table so its row names are the patient IDs, not numbers
df_ibd = df_ibd.set_index('geo_accession')

# 2. Reload the gene matrix fresh (just to be safe!)
df_genes = pd.read_csv(
    file_path, 
    sep='\t', 
    compression='gzip', 
    comment='!', 
    index_col=0
)

# 3. Transpose the gene matrix
df_genes = df_genes.T 

# 4. Filter the gene matrix (This will work perfectly now!)
df_genes = df_genes.loc[df_ibd.index]

# 5. Check the final dimensions
print("Clinical shape:", df_ibd.shape)
print("Gene shape:", df_genes.shape)

In [ ]:
#x = df_genes matrix
#y = response to infliximab column from df_ibd
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    df_genes, #genes
    df_ibd['Response to infliximab'], #what we are trying to predict
    test_size = 0.2, #20% hidden from model
    random_state = 2 #random shuffle is the same each time
)
print(x_train.shape) #output is 96 patients (80%), 54675 genes per patient (x,y) - study guide for model
print(x_test.shape) #output is 25 patients (20%), 54675 genes per patient (x,y) - clues, all hidden from model
#y train = answers for 96 patients (80%)
#y test = answers for 25% hidden

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
   penalty = 'l1',        #uses lasso penalty - deletes genes that have no impact
   solver = 'liblinear',  # math engine needed for L1 penalty
   random_state = 2 #keeps randomisation consistent
)

model.fit(x_train, y_train)
print("Training data analysis finished")

In [ ]:
predictions = model.predict(x_test) #model predicts responses from training data (x test contains 20% hidden from model)

from sklearn.metrics import accuracy_score
final_grade = accuracy_score(y_test, predictions) #answers to 20% hidden from model (y_test) and model predictions

print(f"The model scored: {final_grade * 100: 2f}%") # prints prediction, rounds output to 2dp

In [ ]:
df_results = pd.DataFrame({ #creates dataframe using geness and weights from model
    'genes' : x_train.columns,
    'weight' : model.coef_[0],
})
 
df_results = df_results[df_results['weight'] != 0.0] #keeps weights not equal to 0

print(f"Total surviving genes: {len(df_results)}")             #len counts number of rows in dataframe



In [ ]:
df_results.sort_values('weight', ascending = False).head(10) # sorts the dataframe by ascending weight, displaying the top 10 genes with most impact

In [ ]:
top_10_genes = df_results.sort_values('weight', ascending = False).head(10) # saving top 10 genes to variable

top_10_genes.to_csv("infliximab_top_10_biomarkers.csv", index = False) #exports it to folder, index = false stops row numbers being added to file

print("Exported successfully")

import joblib

joblib.dump(model, "infliximab_lasso_model.pkl")

